# KDA Modifier - Electric Potential

In this notebook, I will aim to modify and adapt the current KDA protocols for 4 node systems, both with and without leakage, for the cases pertaining to electric potential across the cell membrane

First, of course, we need to import all the relevant packages:

In [ ]:
import numpy as np
import networkx as nx
import os
import matplotlib
import sys
from sympy import symbols, latex
from kda import plotting, graph_utils, calculations, diagrams, expressions, ode

From here, we'll construct our 2 4x4 matrices, one where there is no leakage, and another where this is from nodes 2 to 4

In [ ]:
kvals1 = np.array([
    [0, 1, 0, 1],
    [1, 0, 1, 0],
    [0, 1, 0, 1],
    [1, 0, 1, 0]
])

kvals2 = np.array({
    [0, 1, 0, 1],
    [1, 0, 1, 1],
    [0, 1, 0, 1],
    [1, 1, 1, 0]
})

G1 = nx.MultiDiGraph()
graph_utils.generate_edges(G1, kvals1)
G2 = nx.MultiDiGraph()
graph_utils.generate_edges(G2, kvals2)
pos = {0 : [1, 1],
1: [-1, 1],
2: [-1, -1],
3: [1, -1]
}

We'll also define a command to save our figures in our current working directory! Can be customized to save to others. 

In [ ]:
def save_figure(fig, filename):
    cwd = os.getcwd()
    for _ext in ("svg"):
        #I particularly just want to use svgs 
        path = os.path.join(cwd, f"{filename}.{_ext}")
        fig.savefig(path, dpi = 300)

Now we're going to plot the kinetic diagrams:

In [ ]:
plotting.draw_diagrams(G1, pos=pos, font_size = 12, curved_arrows=True)
plotting.draw_diagrams(G2, pos=pos, font_size=12, curved_arrows=True)

#Generating Partials and Directionals

pars = diagrams.generate_partial_diagrams(G2)
dirpars = diagrams.generate_directional_diagrams(G2)

plotting.draw_diagrams(pars, pos=pos, panel=True, panel_scale=1.75, rows=1, cols=8, font_size=12)
plotting.draw_diagrams(dirpars, pos=pos, panel=True, panel_scale=1.75, rows=4, font_size=12, cbt=True)

... collecting cycles

In [ ]:
G1_cycle = graph_utils.find_all_unique_cycles(G1)[0]
G2_cycles = graph_utils.find_all_unique_cycles(G2)

#hand-choosing cycles by putting them into node pairs
G1_order = [0, 1]
G2_orders = [[0, 1], [0, 1], [1, 2]]

#plotting cycles
# G2_cycles_reordered = [G2_cycles[1], G2_cycles[0], G2_cycles[2]]
plotting.draw_cycles(G2, G2_cycles, pos=pos, panel=True, panel_scale =1.75, font_size=12, curved_arrows=True, cbt=True)

Now, we get to the part where we define certain variables. This will be the crux where we implement concepts relating to the potential gradient using SymPy to define variables for substitution. 

In [ ]:
k12, k21, k23, k32, k34, k43, k41, k14, k24, k42 = symbols(
"k12, k21, k23, k32, k34, k43, k41, k14, k24, k42"
)

R_on, R_off, L_on, L_off, k_leak, R_int, R_ext, L_int, L_ext = symbols(
    "R_on, R_off, L_on, L_off, k_leak, R_int, R_ext, L_int, L_ext"
)

#for reference: L stands for Ligand and R is a driving force (i dunno why it's R)
#creating needed constants and symbols 


sub_dict1 = {
    k12: R_off,
    k21: R_on*R_int,
    k23: L_on*L_int,
    k32: L_off,
    k34: L_off,
    k43: L_on*L_ext,
    k41: R_on*R_ext,
    k14: R_off,
}

sub_dict2 = {
    k12: R_off,
    k21: R_on*R_int,
    k23: L_on*L_int,
    k32: L_off,
    k34: L_off,
    k43: L_on*L_ext,
    k41: R_on*R_ext,
    k14: R_off,
    k24: k_leak, #I can essentially set k_leak to whatever the hell I want!
    k42: k_leak,
}

rate_names1 = ["R_on, R_off, L_on, L_off, R_int, R_ext, L_int, L_ext"]
rate_names2 = ["R_on, R_off, L_on, L_off, k_leak, R_int, R_ext, L_int, L_ext"]


Probability Stuffs:

In [ ]:

G1_prob_strs = calculations.calc_state_probs(G1, key='name', output_strings=True)
G2_prob_strs = calculations.calc_state_probs(G2, key='name', output_strings=True)
p1_exp = G2_prob_strs[0]
p1_simple = p1_exp.subs(sub_dict2).simplify()

#For this procedure, I particularly do not need this section.

Net Cycle Flux Expression Manipulation

In [ ]:
J_G1_func = calculations.calc_net_cycle_flux(G1, G1_cycle, order=G1_order, key='name', output_strings=True)
J_G1 = J_G1_func.subs(sub_dict1).simplify()

cycle_dict = {}
for (label, cycle, order) in zip(["a", "b", "c"], G2_cycles, G2_orders):
    func = calculations.calc_net_cycle_flux(G2, cycle, order=order, key='name', output_strings=True)
    func = func.subs(sub_dict2).simplify()
    cycle_dict[label] = {
        "cycle": cycle, 
        "order": order, 
        "func": func}

#While there's other cool stuff we could do with this, it is not within the purview of this tutorial

Operational Flux Analysis

In [ ]:
J_a = cycle_dict["a"]["func"]
J_b = cycle_dict["b"]["func"]
J_c = cycle_dict["c"]["func"]

J_R = (J_a + J_b).simplify()
J_L = (J_a + J_c).simplify()


p1, p2, p3, p4 = G2_prob_strs
J_R_trans = (p1.subs(sub_dict2) * sub_dict2[k12] - p2.subs(sub_dict2) * sub_dict2[k21]).simplify()
J_L_trans = (p2.subs(sub_dict2) * sub_dict2[k23] - p3.subs(sub_dict2) * sub_dict2[k32]).simplify()

(J_R_trans - J_R).simplify() == 0 #Both following equality checks are simply for verification

(J_L_trans - J_L).simplify() == 0

## Analysis!

This part is where we implement the former specific symbols and constants from before

In [ ]:
# to calculate the flux for cycle a for both models
J_G1_lambda = expressions.construct_lambda_funcs(J_G1, rate_names1)
J_a_G2_lambda = expressions.construct_lambda_funcs(cycle_dict["a"]["func"], rate_names2)

# to calculate the operational fluxes for both ligands
J_R_lambda, J_L_lambda = expressions.construct_lambda_funcs([J_R, J_L], rate_names2)

# binding/unbinding rates
R_on = 1e10
R_off = 1e3
L_on = 1e9
L_off = 1e2

# internal/external concentrations
# R_ext / R_int = 10
R_int = 10 ** (-7.5)
R_ext = 10 ** (-6.5)
# L_ext / L_int = 0.631
L_int = 10 ** (-6.9)
L_ext = 10 ** (-7.1)

# create an array of leakage values
leak_arr = np.logspace(-2, 2, 101)

G1_arr = np.zeros(shape=(leak_arr.size,), dtype=np.float64)
G2_arr = np.zeros(shape=(leak_arr.size,), dtype=np.float64)
J_R_arr = np.zeros(shape=(leak_arr.size,), dtype=np.float64)
J_L_arr = np.zeros(shape=(leak_arr.size,), dtype=np.float64)

for i, k_leak in enumerate(leak_arr):
    G1_arr[i] = J_G1_lambda(R_on, R_off, L_on, L_off, R_int, R_ext, L_int, L_ext)
    G2_arr[i] = J_a_G2_lambda(R_on, R_off, L_on, L_off, k_leak, R_int, R_ext, L_int, L_ext)    
    J_R_arr[i] = J_R_lambda(R_on, R_off, L_on, L_off, k_leak, R_int, R_ext, L_int, L_ext)
    J_L_arr[i] = J_L_lambda(R_on, R_off, L_on, L_off, k_leak, R_int, R_ext, L_int, L_ext)

    fig, axs = plt.subplots(3, figsize=(5, 5), sharex=True, tight_layout=True)

#Graphing

axs[0].semilogx(leak_arr, G1_arr, '--', lw=2, color="black", label="G1")
axs[0].semilogx(leak_arr, G2_arr, '-', lw=2, color="orange", label="G2")
axs[0].set_title("Net Cycle Flux: Productive Cycle", fontsize=11)
axs[0].set_ylabel(r"Flux ($s^{-1}$)", fontsize=10)
axs[0].set_ylim(10, 20)
axs[0].legend(bbox_to_anchor=(1, 1))
axs[0].grid(True)

axs[1].semilogx(leak_arr, G1_arr, '--', lw=2, color="black", label="G1")
axs[1].semilogx(leak_arr, J_R_arr, '-', color='red', lw=2, label=r"R")
axs[1].semilogx(leak_arr, J_L_arr, '-', color='blue', lw=2, label=r"L")
axs[1].set_title("Operational Flux", fontsize=11)
axs[1].set_ylabel(r"Flux ($s^{-1}$)", fontsize=10)
axs[1].set_ylim(0, np.max(J_R_arr)*1.1)
axs[1].legend(bbox_to_anchor=(1, 1))
axs[1].grid(True)

axs[2].axhline(y=1, ls='--', color='black', label="1:1")
axs[2].semilogx(leak_arr, J_L_arr/J_R_arr, '-', lw=2, color="purple", label=r"L/R")
axs[2].set_title("Stoichiometry", fontsize=11)
axs[2].set_ylabel("Substrate/Driv. Ion", fontsize=10)
axs[2].set_xlabel(r"$k_{leak}$ ($s^{-1}$)")
axs[2].set_ylim(0, 1.1)
axs[2].legend(bbox_to_anchor=(1, 1))
axs[2].grid(True)
plt.savefig(cwd+"/4wl_all_in_one_plot.png", dpi=300)